# development of z-score plotting framework

NOTE: dataframe arithmetic seems to be simplest if types are not mixed, so this p-o-c:
- does not join the dataframes until calculations are complete
- only adds label/date columns when calcultions are complete
```
2023-05-17 WJP: initial coding, initially just using BCR2G
2023-05-26 WJP: first working version, calculates z-score and zeta score
                TODO: not sure how this works with <mdl values (ie. negatives in the trace_df
                TODO: not sure how this handles missing values in the reference dataset
```

In [1]:
# import some stuff
from pathlib import Path
import importlib
import pandas as pd
import numpy as np
import sys
import os
import re

# for plotting...
import plotly.express as px
import plotly.graph_objects as go

# add the grandparent directory to the path and check that
# py_utility_code directory is accessible
utility_dirname = 'py_utility_code'
parent_dir = os.path.dirname(os.path.abspath('__file__'))
# append the 'grandparent' dir to the path...
grandparentdir = os.path.dirname(parent_dir)
sys.path.append(grandparentdir)
# chech that the py_utility_routines directory is available...
if not (Path(grandparentdir, utility_dirname)).exists():
    print('Unable to access {0}!!'.format(utility_dirname))


# add the requried modules from py_utility_code
# do this using a basic module loader, defined here:
# define a module loader, this assumes the folder containing the modules is on the path

def loadmod(modulename):
    module = importlib.import_module('py_utility_code.{0}'.format(modulename))
    return getattr(module, modulename)

# upbplot_call = loadmod('upbplot_call')
glitter_extract = loadmod('glitter_extract')
# carbon_batchinfo = loadmod('carbon_batchinfo')
extract_labels = loadmod('extract_labels')
element = loadmod('element')

In [2]:
# set path to the edited bcr georem data
# "C:\Users\william.powell\OneDrive - Rio Tinto\QAQC\_headers\bcr-2g-georem-user.csv"
# following is for using BCR:
ref = 'BCR-2G'
referencefile = 'bcr-2g-georem-user.csv'
refmatchtext = r'BCR[-_]?2G'


# following is for NIST610:
# ref = 'NIST-610'
# referencefile = 'nist-610-georem-user.csv'
# refmatchtext = r'NIST[-_]?610'

# ref = 'REF109'
# referencefile = 'REF109-user.csv'
# refmatchtext = r'REF109'

# ref = 'REF265'
# referencefile = 'REF265-user.csv'
# refmatchtext = r'REF265'

# ref = 'REF275'
# referencefile = 'REF275-user.csv'
# refmatchtext = r'REF275'



georemfile = Path(Path.home(), 'OneDrive - Rio Tinto', 'QAQC', '_headers', referencefile)
georem_df = pd.read_csv(georemfile)

In [ ]:

###############################################################################################
# "C:\Users\william.powell\OneDrive - Rio Tinto\QAQC\EB80042285\r_blocks\r_combined_glitter.csv"
# sampledatafile = Path(Path.home(), 'OneDrive - Rio Tinto', 'QAQC',
#                       'EB80042433', 'Zircon_R15392A-94A_T.csv')
# sampledatafile = r"D:\_data\EB80042300\Zircon_R14562A\Zircon_R14562A_T.csv"
# sampledatafile = r"D:\_data\spodumene\20240429-spd\20240429-1_GSD_T.csv"
# sampledatafile = r"D:\_data\titanite\20240426-ttn\20240426-0_T.csv"
# sampledatafile = r"\\aumelfile2\tilab\RIMS\Working\EB80042523\LAICPMS\Spodumene\Spodumene_R15672A-74A\Spodumene_R15672A-74A_T.csv"
sampledatafile = r"C:\Users\junlin.ng\OneDrive - Rio Tinto\QAQC\EB80042717\Fixed_Pb204\Zircon_R16705A_Run2_T.csv"
sampledatafile = Path(sampledatafile)
sampledatafile.exists()
# print(sampledatafile)



True

In [5]:

# path = r"D:\_data\titanite\copied_to_onedrive\20231018-ttn\20231018-0_T.csv"
# sampledatafile = Path(path)
# sampledatafile.exists()


In [6]:


# load the trace element concentration datablock, transpose
trace_df = glitter_extract(sampledatafile, searchkey = 'trace').set_index('Element').T
uncertainty_df = glitter_extract(sampledatafile, searchkey = 'trace_1s').set_index('Element').T
# load the MDL dataset - won't be used until later
mdl_df = glitter_extract(sampledatafile, searchkey = 'mdl').set_index('Element').T

# add a label column for filtering purposes
# list of labels can also be used to build menus in future
labelvalues = []
for d in trace_df.index:
    labelvalues.append(extract_labels(d, label = True))
trace_df.insert(loc = 0, column = 'label', value = labelvalues, allow_duplicates= True)
uncertainty_df.insert(loc = 0, column = 'label', value = labelvalues, allow_duplicates= True)


In [7]:
print(trace_df)

Element                                   label  Li7_ppm  Mg24_ppm  Al27_ppm  \
20250708_GSD-1G-STD73_zrn_B_24-1   GSD-1G-STD73    44.14  22023.66  72731.34   
20250708_GSD-1G-STD73_zrn_B_24-2   GSD-1G-STD73    44.58  21349.00  71502.23   
20250708_BCR-2G-STD64_zrn_B_24-1   BCR-2G-STD64     7.95  20545.57  69606.30   
20250708_BCR-2G-STD64_zrn_B_24-2   BCR-2G-STD64     8.37  20602.90  69626.01   
20250708_BCR-2G-STD64_zrn_B_24-3   BCR-2G-STD64     9.11  20593.48  69237.82   
...                                         ...      ...       ...       ...   
20250708_GJ-REF124_zrn_B_24-34        GJ-REF124    -0.71     -0.39      3.67   
20250708_BCR-2G-STD64_zrn_B_24-33  BCR-2G-STD64     9.44  21945.28  73502.46   
20250708_BCR-2G-STD64_zrn_B_24-34  BCR-2G-STD64     9.00  21525.83  74642.13   
20250708_GSD-1G-STD73_zrn_B_24-33  GSD-1G-STD73    44.59  21392.67  71667.56   
20250708_GSD-1G-STD73_zrn_B_24-34  GSD-1G-STD73    41.71  21262.97  71742.20   

Element                             Si2

In [8]:
# filter dataframe for just the matching ref data
trace_df = trace_df[trace_df['label'].str.contains(refmatchtext)]
uncertainty_df = uncertainty_df[uncertainty_df['label'].str.contains(refmatchtext)]

# now remove the label column so it doesn't mess with the calculations...
trace_df.drop('label', axis = 1, inplace = True)
uncertainty_df.drop('label', axis = 1, inplace = True)

# remove Cl35_ppm from the list, as its not in the BCR georem dataframe
# trace_df = trace_df.drop(columns = ['Cl35_ppm, 'As75_ppm'])
# uncertainty_df = uncertainty_df.drop(columns = ['Cl35_ppm_1s', 'As75_ppm_1s'])
#trace_df = trace_df.drop(columns = [ 'As75_ppm'])
#uncertainty_df = uncertainty_df.drop(columns = ['As75_ppm_1s'])



In [9]:

print(trace_df)

Element                            Li7_ppm  Mg24_ppm  Al27_ppm   Si29_ppm  \
20250708_BCR-2G-STD64_zrn_B_24-1      7.95  20545.57  69606.30  254287.09   
20250708_BCR-2G-STD64_zrn_B_24-2      8.37  20602.90  69626.01  254287.09   
20250708_BCR-2G-STD64_zrn_B_24-3      9.11  20593.48  69237.82  254287.08   
20250708_BCR-2G-STD64_zrn_B_24-4      7.53  20730.19  69478.24  254287.09   
20250708_BCR-2G-STD64_zrn_B_24-5      9.31  20842.90  70357.06  254287.09   
20250708_BCR-2G-STD64_zrn_B_24-6      8.72  20878.63  70267.36  254287.09   
20250708_BCR-2G-STD64_zrn_B_24-7      9.93  21016.30  70687.79  254287.08   
20250708_BCR-2G-STD64_zrn_B_24-8      8.81  20491.40  68909.27  254287.06   
20250708_BCR-2G-STD64_zrn_B_24-9      9.50  20915.93  70606.67  254287.06   
20250708_BCR-2G-STD64_zrn_B_24-10     8.59  20914.82  70987.07  254287.08   
20250708_BCR-2G-STD64_zrn_B_24-11     8.80  20996.29  70836.55  254287.06   
20250708_BCR-2G-STD64_zrn_B_24-12     8.49  21215.97  71116.11  254287.08   

In [10]:
print(uncertainty_df)

Element                            Li7_ppm_1s  Mg24_ppm_1s  Al27_ppm_1s  \
20250708_BCR-2G-STD64_zrn_B_24-1         0.81       839.22      2587.21   
20250708_BCR-2G-STD64_zrn_B_24-2         0.84       841.63      2588.17   
20250708_BCR-2G-STD64_zrn_B_24-3         0.85       841.25      2573.78   
20250708_BCR-2G-STD64_zrn_B_24-4         0.80       846.73      2582.36   
20250708_BCR-2G-STD64_zrn_B_24-5         0.87       853.26      2619.46   
20250708_BCR-2G-STD64_zrn_B_24-6         0.85       854.84      2616.36   
20250708_BCR-2G-STD64_zrn_B_24-7         0.92       866.93      2646.77   
20250708_BCR-2G-STD64_zrn_B_24-8         0.86       845.49      2580.53   
20250708_BCR-2G-STD64_zrn_B_24-9         0.90       874.33      2670.21   
20250708_BCR-2G-STD64_zrn_B_24-10        0.85       874.67      2685.39   
20250708_BCR-2G-STD64_zrn_B_24-11        0.90       895.22      2719.39   
20250708_BCR-2G-STD64_zrn_B_24-12        0.88       905.34      2732.00   
20250708_BCR-2G-STD64_zrn

In [11]:
trace_df.columns

Index(['Li7_ppm', 'Mg24_ppm', 'Al27_ppm', 'Si29_ppm', 'P31_ppm', 'Sc45_ppm',
       'Ti49_ppm', 'V51_ppm', 'Mn55_ppm', 'Fe57_ppm', 'Cu63_ppm', 'Sr88_ppm',
       'Y89_ppm', 'Zr90_ppm', 'Nb93_ppm', 'Mo95_ppm', 'Sn118_ppm', 'Ba137_ppm',
       'La139_ppm', 'Ce140_ppm', 'Pr141_ppm', 'Nd146_ppm', 'Sm147_ppm',
       'Eu153_ppm', 'Gd157_ppm', 'Tb159_ppm', 'Dy163_ppm', 'Ho165_ppm',
       'Er166_ppm', 'Tm169_ppm', 'Yb172_ppm', 'Lu175_ppm', 'Hf178_ppm',
       'Ta181_ppm', 'Pb204_ppm', 'Pb206_ppm', 'Pb207_ppm', 'Pb208_ppm',
       'Th232_ppm', 'U238_ppm'],
      dtype='object', name='Element')

In [12]:
# use the isotope/element names to make a series of georem values
# necessary because georem values are element-based, not isotope based

# the list()[0] wrapper is necessary to change the pd.Series returned by subsetting the dataframe into a scalar value

refvalue = []
refuncertainty = []
for c in trace_df.columns:
    print(c)
    refvalue.append(list(georem_df[georem_df.Item == element(c, symbol = True)].Value)[0])
    refuncertainty.append(list(georem_df[georem_df.Item == element(c, symbol = True)].Uncertainty)[0])


Li7_ppm
Mg24_ppm
Al27_ppm
Si29_ppm
P31_ppm
Sc45_ppm
Ti49_ppm
V51_ppm
Mn55_ppm
Fe57_ppm
Cu63_ppm
Sr88_ppm
Y89_ppm
Zr90_ppm
Nb93_ppm
Mo95_ppm
Sn118_ppm
Ba137_ppm
La139_ppm
Ce140_ppm
Pr141_ppm
Nd146_ppm
Sm147_ppm
Eu153_ppm
Gd157_ppm
Tb159_ppm
Dy163_ppm
Ho165_ppm
Er166_ppm
Tm169_ppm
Yb172_ppm
Lu175_ppm
Hf178_ppm
Ta181_ppm
Pb204_ppm
Pb206_ppm
Pb207_ppm
Pb208_ppm
Th232_ppm
U238_ppm


In [13]:
# alternative version - just plot Li7
# refvalue = []
# refuncertainty = []
# c = 'Li7'
# refvalue.append(list(georem_df[georem_df.Item == element(c, symbol = True)].Value)[0])
# refuncertainty.append(list(georem_df[georem_df.Item == element(c, symbol = True)].Uncertainty)[0])

# # remove Cl from the list, as its not in the BCR georem dataframe
# trace_df = trace_df.drop(columns = ['Cl35_ppm', 'As75_ppm'])
# uncertainty_df = uncertainty_df.drop(columns = ['Cl35_ppm_1s', 'As75_ppm_1s'])



we now have lists containing the reference values and uncertainties that can be used for vector calculations with the dataframe

In [14]:
# make a new df with the z-score values
zscore_df = (trace_df - refvalue) / refuncertainty
# rename column headers
zscore_df.columns = zscore_df.columns.str.replace('_ppm', '_zscore')

In [15]:
zscore_df.columns

Index(['Li7_zscore', 'Mg24_zscore', 'Al27_zscore', 'Si29_zscore', 'P31_zscore',
       'Sc45_zscore', 'Ti49_zscore', 'V51_zscore', 'Mn55_zscore',
       'Fe57_zscore', 'Cu63_zscore', 'Sr88_zscore', 'Y89_zscore',
       'Zr90_zscore', 'Nb93_zscore', 'Mo95_zscore', 'Sn118_zscore',
       'Ba137_zscore', 'La139_zscore', 'Ce140_zscore', 'Pr141_zscore',
       'Nd146_zscore', 'Sm147_zscore', 'Eu153_zscore', 'Gd157_zscore',
       'Tb159_zscore', 'Dy163_zscore', 'Ho165_zscore', 'Er166_zscore',
       'Tm169_zscore', 'Yb172_zscore', 'Lu175_zscore', 'Hf178_zscore',
       'Ta181_zscore', 'Pb204_zscore', 'Pb206_zscore', 'Pb207_zscore',
       'Pb208_zscore', 'Th232_zscore', 'U238_zscore'],
      dtype='object', name='Element')

In [16]:
# make a new df with the zeta score values
# two part calculation...
numerator = (trace_df - refvalue)
denominator = np.sqrt(np.power(refuncertainty, 2) + np.power(uncertainty_df, 2))
# column names don't match between trace and uncertainties - causes issues with the arithmetic!
denominator.columns = numerator.columns
zetascore_df = numerator / denominator
# rename column headers
zetascore_df.columns = zetascore_df.columns.str.replace('_ppm', '_zeta')

In [17]:
# ... join it all together
trace_df = trace_df.merge(uncertainty_df, left_index = True, right_index= True)
trace_df = trace_df.merge(zscore_df, left_index = True, right_index = True)
trace_df = trace_df.merge(zetascore_df, left_index = True, right_index = True)
# add label back in
labelvalues = []
for d in trace_df.index:
    labelvalues.append(extract_labels(d, label = True))
trace_df.insert(loc = 0, column = 'label', value = labelvalues, allow_duplicates= True)
# name the index column
trace_df.index.name = 'Analysis_#'
trace_df.reset_index(inplace=True)

In [18]:
trace_df.columns

Index(['Analysis_#', 'label', 'Li7_ppm', 'Mg24_ppm', 'Al27_ppm', 'Si29_ppm',
       'P31_ppm', 'Sc45_ppm', 'Ti49_ppm', 'V51_ppm',
       ...
       'Yb172_zeta', 'Lu175_zeta', 'Hf178_zeta', 'Ta181_zeta', 'Pb204_zeta',
       'Pb206_zeta', 'Pb207_zeta', 'Pb208_zeta', 'Th232_zeta', 'U238_zeta'],
      dtype='object', name='Element', length=162)

In [19]:
trace_df.to_clipboard()
# set a path for saving the file
# outfile = Path(Path.home(), 'OneDrive - Rio Tinto', 'QAQC', 'complete', 'EB80042259', 'Zircon_R14090A-91A_bcr_Z.csv')
# trace_df.to_csv(outfile)

In [20]:
# alternatively, can join the z-score and zeta-score dataframes into the original complete dataset
make_combined = False
if make_combined:
    combined_df = trace_df.merge(uncertainty_df.drop('label', axis = 1), left_index = True, right_index = True)
    combined_df = combined_df.merge(mdl_df, left_index = True, right_index = True)
    combined_df = combined_df.merge(zscore_df, how = 'left', left_index = True, right_index = True)
    # combined_df.shape
    combined_df = combined_df.merge(zetascore_df, how = 'left', left_index = True, right_index = True)
    combined_df.index.name = 'Analysis_#'
    combined_df.to_clipboard()

In [21]:
# set the analyte to plot  [ try using a dash/pxwidget here ]
analyte = 'Li7'

# concentration in this case
fig = px.scatter(trace_df, x = 'Analysis_#', y = analyte + '_ppm', error_y = analyte + '_ppm_1s')

# add the reference +/- uncertainty lines
reference = georem_df[georem_df['Item'] == element(analyte, symbol = True)]
fig.add_hline(y = reference.Value.item(), line_color = 'gray')
fig.add_hline(y = reference.Value.item() - reference.Uncertainty.item(), line_dash = 'dash', line_color = 'grey')
fig.add_hline(y = reference.Value.item() + reference.Uncertainty.item(), line_dash = 'dash', line_color = 'grey')

fig.show()

In [22]:
# zscore, multi-element by analysis
# plotvar = '_zscore'
plotvar = '_zeta'

fig = go.Figure()
for el in trace_df.columns[trace_df.columns.str.contains(plotvar)]:
    fig.add_trace(go.Scatter(x = trace_df['Analysis_#'], y = trace_df[el], mode = 'markers', name = el.replace(plotvar, '')))
fig.show()

In [23]:

filesuffix = '_' + ref + '_' + plotvar
savepath = Path(sampledatafile.parent, sampledatafile.name.replace('.csv', filesuffix + '_by_analysis.html'))
# fig.write_image(savepath)
fig.write_html(savepath)


In [24]:

savepath = Path(sampledatafile.parent, sampledatafile.name.replace('.csv', filesuffix + '_by_analysis.png'))
fig.write_image(savepath)


In [25]:
fig = go.Figure()
for ii, row in trace_df.iterrows():
    # hoverlabel
    fig.add_trace(go.Scatter(
        x = row[row.index.str.contains(plotvar)].index.str.replace(plotvar, ''),
        y = row[row.index.str.contains(plotvar)], mode = 'markers',
        name = row['Analysis_#'],
        ))
fig.show()

In [26]:
savepath = Path(sampledatafile.parent, sampledatafile.name.replace('.csv', filesuffix + '_by_analyte.html'))
# fig.write_image(savepath)
fig.write_html(savepath)


In [27]:

savepath = Path(sampledatafile.parent, sampledatafile.name.replace('.csv', filesuffix + '_by_analyte.png'))
fig.write_image(savepath)
